# Predict survival on the Titanic and get familiar with ML basics
- https://www.kaggle.com/competitions/titanic/overview

# Data Sets
- https://www.kaggle.com/competitions/titanic/data

In [70]:
import pandas as pd
import numpy as np

## Features

In [71]:
# Create a dictionary with the data
data_vars = {
    'Variable': ['survival', 'pclass', 'sex', 'Age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked'],
    'Definition': ['Survival', 'Ticket class', 'Sex', 'Age in years', '# of siblings / spouses aboard the Titanic', 
                   '# of parents / children aboard the Titanic', 'Ticket number', 'Passenger fare', 
                   'Cabin number', 'Port of Embarkation'],
    'Key': ['0 = No, 1 = Yes', '1 = 1st, 2 = 2nd, 3 = 3rd', '', '', '', '', '', '', '', 
            'C = Cherbourg, Q = Queenstown, S = Southampton']
}

# Create the DataFrame
df_vars = pd.DataFrame(data_vars)

# Display the DataFrame
df_vars

,Variable,Definition,Key
0,survival,Survival,"0 = No, 1 = Yes"
1,pclass,Ticket class,"1 = 1st, 2 = 2nd, 3 = 3rd"
2,sex,Sex,
3,Age,Age in years,
4,sibsp,# of siblings / spouses aboard the Titanic,
5,parch,# of parents / children aboard the Titanic,
6,ticket,Ticket number,
7,fare,Passenger fare,
8,cabin,Cabin number,
9,embarked,Port of Embarkation,"C = Cherbourg, Q = Queenstown, S = Southampton"


## Load training data

In [72]:
train_data = pd.read_csv('data/train.csv')
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Explore training data

In [73]:
train_data.shape

(891, 12)

In [74]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [75]:
train_data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## Create a copy of the training data

In [76]:
df = train_data.copy()

In [77]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [78]:
df['Ticket'].value_counts()

Ticket
347082              7
1601                7
CA. 2343            7
3101295             6
CA 2144             6
                   ..
PC 17590            1
17463               1
330877              1
373450              1
STON/O2. 3101282    1
Name: count, Length: 681, dtype: int64

### Remove unecessary features
- Cabin has 687 missing values
- Ticket has 681 value counts
- Name is unique

In [79]:
df.drop(['Name','Ticket','Cabin'], axis=1, inplace=True)

### Convert Sex to number
- male = 0
- female = 1

In [80]:
filt_male = (df['Sex'] == 'male')
filt_female = (df['Sex'] == 'female')
df.loc[filt_male, 'Sex'] = 0
df.loc[filt_female, 'Sex'] = 1

In [81]:
df.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,0,22.0,1,0,7.2500,S
1,2,1,1,1,38.0,1,0,71.2833,C
2,3,1,3,1,26.0,0,0,7.9250,S
3,4,1,1,1,35.0,1,0,53.1000,S
4,5,0,3,0,35.0,0,0,8.0500,S


### Make missing embark 'U'

In [82]:
# Replace NaN values in the 'Embarked' column with 'U'
df['Embarked'] = df['Embarked'].fillna('U')

In [83]:
# Check for NaN
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Sex              0
Age            177
SibSp            0
Parch            0
Fare             0
Embarked         0
dtype: int64

In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
 8   Embarked     891 non-null    object 
dtypes: float64(2), int64(5), object(2)
memory usage: 62.8+ KB


### OneHotEncoder for Embarked values

In [85]:
# Embark values will need to be converted to numbers before we can run machine learning algorithms on the data.
from sklearn.preprocessing import OneHotEncoder

# Create OneHotEncoder instance
encoder = OneHotEncoder(sparse_output=False)

# Apply OneHotEncoder to Embarked column
encoded_embarked = encoder.fit_transform(df[['Embarked']])

# Get the category names from the encoder
embarked_categories = encoder.categories_[0]

# Create column names with the prefix 'Embarked_'
encoded_column_names = [f'Embarked_{category}' for category in embarked_categories]

# Create a DataFrame with the encoded columns
embarked_encoded_df = pd.DataFrame(encoded_embarked, columns=encoded_column_names, index=df.index)

# Join the encoded columns back to the original DataFrame
df = pd.concat([df, embarked_encoded_df], axis=1)

# Drop the original 'Embarked' column:
df = df.drop('Embarked', axis=1)

# Display the first few rows of the result
df.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_U
0,1,0,3,0,22.0,1,0,7.2500,0.0,0.0,1.0,0.0
1,2,1,1,1,38.0,1,0,71.2833,1.0,0.0,0.0,0.0
2,3,1,3,1,26.0,0,0,7.9250,0.0,0.0,1.0,0.0
3,4,1,1,1,35.0,1,0,53.1000,0.0,0.0,1.0,0.0
4,5,0,3,0,35.0,0,0,8.0500,0.0,0.0,1.0,0.0


### Fill NaN values for Age with median

In [86]:
df['Age'].median()

28.0

In [87]:
df['Age'] = df['Age'].fillna(df['Age'].median())

In [88]:
df.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked_C     0
Embarked_Q     0
Embarked_S     0
Embarked_U     0
dtype: int64

In [89]:
# Then explicitly convert the column to integer type
df['Sex'] = df['Sex'].astype(int)

In [90]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    int64  
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
 8   Embarked_C   891 non-null    float64
 9   Embarked_Q   891 non-null    float64
 10  Embarked_S   891 non-null    float64
 11  Embarked_U   891 non-null    float64
dtypes: float64(6), int64(6)
memory usage: 83.7 KB


In [91]:
# Create a function to do the above
def clean_data(my_df):
    # Drop unneeded columns
    my_df.drop(['Name','Ticket','Cabin'], axis=1, inplace=True)
    
    # Convert m/f to number
    filt_male = (my_df['Sex'] == 'male')
    filt_female = (my_df['Sex'] == 'female')
    my_df.loc[filt_male, 'Sex'] = 0
    my_df.loc[filt_female, 'Sex'] = 1
    
    # Replace NaN values in the 'Embarked' column with 'U'
    my_df['Embarked'] = my_df['Embarked'].fillna('U')

    # Embark values will need to be converted to numbers before we can run machine learning algorithms on the data.
    from sklearn.preprocessing import OneHotEncoder
    
    # Create OneHotEncoder instance
    encoder = OneHotEncoder(sparse_output=False)
    
    # Apply OneHotEncoder to Embarked column
    encoded_embarked = encoder.fit_transform(my_df[['Embarked']])
    
    # Get the category names from the encoder
    embarked_categories = encoder.categories_[0]
    
    # Create column names with the prefix 'Embarked_'
    encoded_column_names = [f'Embarked_{category}' for category in embarked_categories]
    
    # Create a DataFrame with the encoded columns
    embarked_encoded_df = pd.DataFrame(encoded_embarked, columns=encoded_column_names, index=my_df.index)
    
    # Join the encoded columns back to the original DataFrame
    my_df = pd.concat([my_df, embarked_encoded_df], axis=1)
    
    # Drop the original 'Embarked' column:
    my_df = my_df.drop('Embarked', axis=1)

    # Fill missing age values with median
    my_df['Age'] = my_df['Age'].fillna(my_df['Age'].median())

    # Then explicitly convert the column to integer type
    my_df['Sex'] = my_df['Sex'].astype(int)

    return my_df

## Split X, y

In [92]:
X = df.drop(['Survived'], axis=1)

In [93]:
y = df['Survived']

## train_test_split

In [94]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [95]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((712, 11), (179, 11), (712,), (179,))

# Create the model

# Experimenting with different models
* [LinearSVC](https://scikit-learn.org/stable/modules/svm.html#classification)
* [KNeighborsClassifier](https://scikit-learn.org/stable/modules/neighbors.html) (also known as K-Nearest Neighbors or KNN)
* [SVC](https://scikit-learn.org/stable/modules/svm.html#classification) (also known as support vector classifier, a form of [support vector machine](https://en.wikipedia.org/wiki/Support-vector_machine))
* [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) (despite the name, this is actually a classifier)
* [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) (an ensemble method and what we used above)

In [96]:
# Import RandomForestClassifier form sklearn's ensemble module
from sklearn.ensemble import RandomForestClassifier

# Import LinearSVC from sklearn's svm module
from sklearn.svm import LinearSVC

# Import KNeighborsClassifier from sklearn's neighbors module
from sklearn.neighbors import KNeighborsClassifier

# Import SVC from sklearn's svm module
from sklearn.svm import SVC

# Import LogisticRegression from sklearn's linear_model module
from sklearn.linear_model import LogisticRegression

In [97]:
# Create a dictionary called models which contains all of the classification models we've imported
# Make sure the dictionary is in the same format as example_dict
# The models dictionary should contain 5 models
models = {"LinearSVC": LinearSVC(),
          "KNN": KNeighborsClassifier(),
          "SVC": SVC(),
          "LogisticRegression": LogisticRegression(),
          "RandomForestClassifier": RandomForestClassifier()}

# Create an empty dictionary called results
results = {}

In [98]:
# Loop through the models dictionary items, fitting the model on the training data
# and appending the model name and model score on the test data to the results dictionary
for model_name, model in models.items():
    model.fit(X_train, y_train)
    results[model_name] = model.score(X_test, y_test)
    
results

/home/wsl2/Machine_Learning/kaggle/competitions/Titanic/env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'LinearSVC': 0.776536312849162,
 'KNN': 0.6703910614525139,
 'SVC': 0.7039106145251397,
 'LogisticRegression': 0.776536312849162,
 'RandomForestClassifier': 0.8435754189944135}

## Tuning with Hyperparameters

In [99]:
# # Example of adjusting hyperparameters computationally (recommended)

# from sklearn.model_selection import RandomizedSearchCV

# # Define a grid of hyperparameters
# grid = {"n_estimators": [10, 100, 200, 500, 1000, 1200],
#         "max_depth": [None, 5, 10, 20, 30],
#         "max_features": ["log2", "sqrt"],
#         "min_samples_split": [2, 4, 6],
#         "min_samples_leaf": [1, 2, 4]}

# # Set n_jobs to -1 to use all cores (NOTE: n_jobs=-1 is broken as of 8 Dec 2019, using n_jobs=1 works)
# clf = RandomForestClassifier(n_jobs=1)

# # Setup RandomizedSearchCV
# rs_clf = RandomizedSearchCV(estimator=clf,
#                             param_distributions=grid,
#                             n_iter=10, # try 10 models total
#                             cv=5, # 5-fold cross-validation
#                             verbose=2) # print out results

# # Fit the RandomizedSearchCV version of clf
# rs_clf.fit(X_train, y_train);

# # Find the best hyperparameters
# print(rs_clf.best_params_)

# # Scoring automatically uses the best hyperparameters
# rs_clf.score(X_test, y_test)

In [100]:
# # Example of adjusting hyperparameters computationally

# from sklearn.model_selection import GridSearchCV

# # Define a grid of hyperparameters
# grid = {"n_estimators": [100, 200, 500, 1000, 1200],
#         "max_depth": [5, 10],
#         "max_features": ["log2", "sqrt"],
#         "min_samples_split": [2, 4, 6],
#         "min_samples_leaf": [1, 2, 4]}

# # Set n_jobs to -1 to use all cores (NOTE: n_jobs=-1 is broken as of 8 Dec 2019, using n_jobs=1 works)
# clf = RandomForestClassifier(n_jobs=1)

# # Setup RandomizedSearchCV
# rs_clf = GridSearchCV(estimator=clf,
#                       param_grid=grid,
#                       cv=5, # 5-fold cross-validation
#                       verbose=2) # print out results

# # Fit the RandomizedSearchCV version of clf
# rs_clf.fit(X_train, y_train);

# # Find the best hyperparameters
# print(rs_clf.best_params_)

# # Scoring automatically uses the best hyperparameters
# rs_clf.score(X_test, y_test)

In [101]:
# Use the best parameters found above to create the model
tuned_clf = RandomForestClassifier(n_estimators=100, min_samples_split=2, min_samples_leaf=1, max_features='log2', max_depth=5)

In [119]:
# Try a non tuned model for a second submission
non_tuned_clf = RandomForestClassifier()
non_tuned_clf.fit(X_train, y_train)

RandomForestClassifier()

In [102]:
# Fit the tuned model
tuned_clf.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, max_features='log2')

# Export the trained model

In [103]:
# Import the dump and load functions from the joblib library
from joblib import dump, load

# Use the dump function to export the trained model to file
dump(tuned_clf, "trained-classifier.joblib")

['trained-classifier.joblib']

# Import the Test Data

In [104]:
test_data = pd.read_csv('data/test.csv')

In [105]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## Clean the test data

In [106]:
test_data.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [107]:
clean_test = clean_data(test_data)
clean_test.head()

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,892,3,0,34.5,0,0,7.8292,0.0,1.0,0.0
1,893,3,1,47.0,1,0,7.0000,0.0,0.0,1.0
2,894,2,0,62.0,0,0,9.6875,0.0,1.0,0.0
3,895,3,0,27.0,0,0,8.6625,0.0,0.0,1.0
4,896,3,1,22.0,1,1,12.2875,0.0,0.0,1.0


### Fill missing Fare with median

In [108]:
clean_test['Fare'].median()

14.4542

In [109]:
clean_test['Fare'] = clean_test['Fare'].fillna(clean_test['Fare'].median())

### Add Embarked_U column

In [110]:
clean_test['Embarked_U'] = 0.0

In [111]:
clean_test.isna().sum()

PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked_C     0
Embarked_Q     0
Embarked_S     0
Embarked_U     0
dtype: int64

In [112]:
clean_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Sex          418 non-null    int64  
 3   Age          418 non-null    float64
 4   SibSp        418 non-null    int64  
 5   Parch        418 non-null    int64  
 6   Fare         418 non-null    float64
 7   Embarked_C   418 non-null    float64
 8   Embarked_Q   418 non-null    float64
 9   Embarked_S   418 non-null    float64
 10  Embarked_U   418 non-null    float64
dtypes: float64(6), int64(5)
memory usage: 36.1 KB


# Make Predictions with the model

In [113]:
# Make predictions on test data and save them
y_preds = tuned_clf.predict(clean_test)

In [116]:
y_preds

array([0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0,
       0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1,
       1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

## Format the prediction output

In [117]:
df_preds = pd.DataFrame()
df_preds['PassengerId'] = clean_test['PassengerId']
df_preds['Survived'] = y_preds
df_preds

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [118]:
# Export prediction data
df_preds.to_csv("data/Titanic_test_predictions.csv", index=False)

In [120]:
# Make predictions on test data and save them
non_tuned_y_preds = non_tuned_clf.predict(clean_test)

In [121]:
non_tuned_df_preds = pd.DataFrame()
non_tuned_df_preds['PassengerId'] = clean_test['PassengerId']
non_tuned_df_preds['Survived'] = non_tuned_y_preds
non_tuned_df_preds

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [122]:
# Export non tuned prediction data
non_tuned_df_preds.to_csv("data/Titanic_test_predictionsNT.csv", index=False)